# PlantSeg -- Array 2nd-Revision Notebook D2: ViT/Swin seed-42 collapse-vs-scatter diagnostic
## (one additional free diagnostic cell before committing to a full retrain)

**Why this notebook exists.** Notebook D proved, with real accuracy numbers, that the ViT-B/16
and Swin-Tiny seed-42 checkpoints are **not** recoverable by changing the installed `timm`
version: every real version (1.0.9 down to 0.9.2) produced byte-identical quick-subset accuracy
(ViT: 9.17% every time; Swin: 0.00% every time). A separate offline check of the raw checkpoint
tensors (outside Kaggle, no GPU needed) found **no corruption** -- no NaN/Inf, no all-zero
layers, and per-layer weight magnitudes closely matching the known-good seed-123/seed-7
checkpoints for the same two architectures. Class-label mapping, training image resolution, and
checkpoint key-structure have all separately been confirmed to match between training and eval.

**What's still open.** Magnitude alone cannot tell a genuinely-trained head apart from a head
that was *never actually updated* (both look like small, unremarkable numbers). And a flat
9.17%/0.00% on a single 120-image quick subset doesn't by itself distinguish "this model learned
nothing at all" from "this model learned something, just not usefully, and got unlucky on this
particular subset." This notebook adds exactly one more cheap check before committing to a full
seed-42 retrain: does each model's *behavior* look like scattered-but-wrong guessing, or a
degenerate collapse onto one (or a few) classes -- and does it do any better on images from its
**own training split** than on held-out validation images?

**Logic:**
- If accuracy on a sample of **training-split** images is also near chance, the checkpoint did
  not even memorize the data it was fitted on -- strong evidence of a training-time problem
  (e.g. a best-epoch-selection bug, a frozen/never-updated head, an optimizer that never
  stepped), not an evaluation-side issue of any kind.
- If predictions collapse onto one or two classes (e.g. the head always predicts the single
  most common disease class, or always predicts class 0), that's the classic signature of a
  head whose weights never moved far from initialization, or that effectively ignores its input.
- If predictions are scattered across many classes but still mostly wrong, and train accuracy is
  also near chance, that rules out a "frozen head" explanation specifically but still points to
  the same conclusion: this checkpoint does not encode a working classifier for this task.

**Scope.** Classifier seed-42 checkpoints for ViT-B/16 and Swin-Tiny only. Uses the single timm
version (`1.0.9`) already confirmed to load every key with zero missing/unexpected -- no version
ladder needed here, since Notebook D already proved version has no effect on the result.

## Estimated runtime
Two models x (120 val images + 120 train images) = 480 forward passes total. A few minutes.

## Setup

In [1]:
import os, json, glob, random, warnings, io, zipfile, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.9"],
               check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
import timm
print("timm:", timm.__version__)

Device: cuda
timm: 1.0.9


## Config -- classifier data only, now with a train-split sample too

In [2]:
# ============ CONFIG ============
def find_data_root():
    for cand in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"),
                 Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): return cand
    hits = glob.glob("/kaggle/input/**/Metadata.csv", recursive=True)
    assert hits, "Metadata.csv not found anywhere under /kaggle/input -- attach the PlantSeg dataset."
    return Path(hits[0]).parent

DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT)

OUTDIR = Path("/kaggle/working/r2_notebookD2_results"); OUTDIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE_CLS_TF = 224
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta.columns = [c.strip() for c in meta.columns]
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["label"] = meta["Disease"]
CLASS_NAMES = sorted(meta["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}
idx2label = {i: c for c, i in label2idx.items()}

train_df = meta[meta.split=="train"].reset_index(drop=True)
val_df   = meta[meta.split=="val"].reset_index(drop=True)
CHANCE = 1.0 / NUM_CLASSES
print(f"classes={NUM_CLASSES}  train={len(train_df)}  val={len(val_df)}  chance_acc={CHANCE:.4f}")

# majority class in the training set -- used below to check for "predicts the prior" collapse
_train_class_counts = train_df["label"].value_counts()
MAJORITY_CLASS = _train_class_counts.index[0]
print(f"Training-set majority class: {MAJORITY_CLASS!r} ({_train_class_counts.iloc[0]} / {len(train_df)} "
      f"= {_train_class_counts.iloc[0]/len(train_df):.4f} of all training images)")

def cls_ckpt_candidates(name, seed):
    return [f"{name}_best.pt"] if seed == 42 else [f"{name}_seed{seed}_best.pt"]
print("Config ready.")

DATA_ROOT: /kaggle/input/datasets/alenmaruf/plantseg/plantseg
classes=115  train=5367  val=846  chance_acc=0.0087
Training-set majority class: 'citrus canker' (225 / 5367 = 0.0419 of all training images)
Config ready.


## Robust checkpoint finder (identical to Notebooks B/C/D)

In [3]:
# ============ ROBUST CHECKPOINT FINDER ============
def _rezip_dir_as_pt(dirpath: Path):
    root_name = dirpath.name
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_STORED) as zf:
        for f in dirpath.rglob("*"):
            if f.is_file():
                zf.write(f, arcname=f"{root_name}/{f.relative_to(dirpath).as_posix()}")
    buf.seek(0)
    return buf

_ckpt_cache = {}

def find_and_load_checkpoint(filename_candidates, search_root="/kaggle/input"):
    key = tuple(filename_candidates)
    if key in _ckpt_cache:
        return _ckpt_cache[key]
    for fname in filename_candidates:
        hits = glob.glob(f"{search_root}/**/{fname}", recursive=True)
        for h in hits:
            hp = Path(h)
            try:
                state = torch.load(hp, map_location="cpu")
                print(f"    [find_and_load] loaded {hp}")
                _ckpt_cache[key] = state; return state
            except Exception as e:
                print(f"    [find_and_load] {hp} exists but torch.load failed directly ({type(e).__name__}: {e});"
                      f" checking for an auto-extracted folder instead...")
        stem = Path(fname).stem
        dir_hits = [d for d in glob.glob(f"{search_root}/**/{stem}", recursive=True) if Path(d).is_dir()]
        for d in dir_hits:
            dp = Path(d)
            if (dp / "data.pkl").exists():
                print(f"    [find_and_load] found extracted checkpoint folder {dp}; re-zipping in memory")
                state = torch.load(_rezip_dir_as_pt(dp), map_location="cpu")
                _ckpt_cache[key] = state; return state
            inner = dp / stem
            if inner.is_dir() and (inner / "data.pkl").exists():
                print(f"    [find_and_load] found nested extracted checkpoint folder {inner}; re-zipping in memory")
                state = torch.load(_rezip_dir_as_pt(inner), map_location="cpu")
                _ckpt_cache[key] = state; return state
    raise FileNotFoundError(f"None of {filename_candidates} found (as file or extracted folder) under {search_root}")

def _unwrap_state(raw):
    if isinstance(raw, dict):
        for key in ("state_dict", "model_state_dict", "model", "best_state"):
            if key in raw and isinstance(raw[key], dict):
                raw = raw[key]; break
    return {k.replace("module.", ""): v for k, v in raw.items()}

def apply_state(model, raw):
    raw = _unwrap_state(raw)
    missing, unexpected = model.load_state_dict(raw, strict=False)
    print(f"    [apply_state] missing={len(missing)} unexpected={len(unexpected)}")
    total_model_keys = len(model.state_dict())
    if total_model_keys and len(missing) >= total_model_keys * 0.5:
        raise RuntimeError(f">=50% of keys unmatched (missing={len(missing)}, unexpected={len(unexpected)}) "
                            f"-- checkpoint did not load; refusing to report numbers from random weights")
    return model

TF_MODEL_IDS = {"ViTBase16": "vit_base_patch16_224", "SwinTiny": "swin_tiny_patch4_window7_224"}
print("Checkpoint finder ready.")

Checkpoint finder ready.


## Collapse-vs-scatter diagnostic

For each model: sample 120 train images + 120 val images (both fixed seed=42, same sampling
scheme as Notebook D's quick subset so results are directly comparable). For every image record
the true label, the predicted label, whether the true label is in the top-5, and the model's max
softmax probability (confidence). Then summarize: top-1 / top-5 accuracy on each split, mean
confidence, and the predicted-class distribution (collapsed onto a handful of classes, or
scattered across most of the 115).

In [4]:
# ============ RUN DIAGNOSTIC FOR BOTH MODELS ============
_tf = T.Compose([T.Resize((IMG_SIZE_CLS_TF, IMG_SIZE_CLS_TF)), T.ToTensor(), T.Normalize(MEAN, STD)])

def sample_df(df, n, seed=42):
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(df), size=min(n, len(df)), replace=False)
    return df.iloc[idx].reset_index(drop=True)

def run_diagnostic(name, df, split_name, model):
    model.eval()
    rows = []
    with torch.no_grad():
        for r in df.itertuples():
            x = _tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)[0]
            top5 = torch.topk(probs, k=5).indices.tolist()
            pred = top5[0]
            true_idx = label2idx[r.label]
            rows.append({
                "split": split_name, "true_label": r.label, "pred_label": idx2label[pred],
                "correct": int(pred == true_idx), "top5_correct": int(true_idx in top5),
                "max_prob": float(probs[pred]),
            })
    out = pd.DataFrame(rows)
    acc = out["correct"].mean(); top5acc = out["top5_correct"].mean(); mean_conf = out["max_prob"].mean()
    pred_counts = out["pred_label"].value_counts()
    n_unique_preds = pred_counts.shape[0]
    top_pred, top_pred_frac = pred_counts.index[0], pred_counts.iloc[0] / len(out)
    print(f"  [{name} / {split_name}]  n={len(out)}  top1_acc={acc:.4f}  top5_acc={top5acc:.4f}  "
          f"mean_confidence={mean_conf:.4f}")
    print(f"      unique predicted classes used: {n_unique_preds} / {NUM_CLASSES}   "
          f"most-predicted class: {top_pred!r} ({top_pred_frac:.4f} of all predictions)"
          f"{'  <-- matches training-set majority class!' if top_pred == MAJORITY_CLASS else ''}")
    print(f"      top-5 most-predicted classes:\n{pred_counts.head(5).to_string()}")
    return out, {"model": name, "split": split_name, "n": len(out), "top1_acc": acc, "top5_acc": top5acc,
                 "mean_confidence": mean_conf, "n_unique_preds": n_unique_preds,
                 "top_pred_class": top_pred, "top_pred_frac": top_pred_frac,
                 "top_pred_is_majority_class": bool(top_pred == MAJORITY_CLASS)}

all_predictions = []
summary_rows = []
for name in ["ViTBase16", "SwinTiny"]:
    print(f"\n=== {name} seed42 ===")
    raw = find_and_load_checkpoint(cls_ckpt_candidates(name, 42))
    model = timm.create_model(TF_MODEL_IDS[name], pretrained=False, num_classes=NUM_CLASSES).to(device)
    model = apply_state(model, raw)

    train_sample = sample_df(train_df, 120, seed=42)
    val_sample   = sample_df(val_df, 120, seed=42)  # identical sampling to Notebook D's QUICK_DF

    train_preds, train_summary = run_diagnostic(name, train_sample, "train", model)
    val_preds, val_summary     = run_diagnostic(name, val_sample, "val", model)

    all_predictions.append(train_preds); all_predictions.append(val_preds)
    summary_rows.append(train_summary); summary_rows.append(val_summary)

    del model; torch.cuda.empty_cache()

predictions_df = pd.concat(all_predictions, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)
predictions_df.to_csv(OUTDIR/"vit_swin_seed42_collapse_diagnostic_predictions.csv", index=False)
summary_df.to_csv(OUTDIR/"vit_swin_seed42_collapse_diagnostic_summary.csv", index=False)
print("\n" + "="*70)
print("SUMMARY (train vs val, both models):")
print(summary_df.to_string(index=False))


=== ViTBase16 seed42 ===
    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/ViTBase16_best.pt
    [apply_state] missing=0 unexpected=0
  [ViTBase16 / train]  n=120  top1_acc=0.1250  top5_acc=0.3250  mean_confidence=0.0791
      unique predicted classes used: 51 / 115   most-predicted class: 'strawberry anthracnose' (0.0500 of all predictions)
      top-5 most-predicted classes:
pred_label
strawberry anthracnose    6
corn gray leaf spot       6
strawberry leaf scorch    6
wheat stripe rust         6
wheat head scab           5
  [ViTBase16 / val]  n=120  top1_acc=0.0917  top5_acc=0.2500  mean_confidence=0.0745
      unique predicted classes used: 58 / 115   most-predicted class: 'blueberry mummy berry' (0.0500 of all predictions)
      top-5 most-predicted classes:
pred_label
blueberry mummy berry            6
wheat powdery mildew             6
carrot alternaria leaf blight    6
wheat stripe

## Wrap-up -- plain-language read of the result + zip for download

In [5]:
print("="*70)
print("HOW TO READ THIS:")
print("- If top1_acc on 'train' is ALSO near chance (~0.0087) for a model, that model's checkpoint")
print("  did not even memorize its own training images -- points to a training-time problem")
print("  (bad best-epoch checkpointing, a head that never updated, etc.), not an eval-side issue.")
print("- If 'n_unique_preds' is tiny (e.g. 1-3 out of 115) and/or 'top_pred_is_majority_class' is")
print("  True, the model has collapsed onto predicting (close to) a constant output -- classic")
print("  sign of a head that never learned to use its input.")
print("- If predictions are scattered across many classes but still mostly wrong on BOTH splits,")
print("  that rules out simple collapse but still means this checkpoint is not a working")
print("  classifier for this task.")
print("="*70)

import zipfile as _zf
zip_path = Path("/kaggle/working/plantseg_r2_notebookD2_results.zip")
with _zf.ZipFile(zip_path, "w", _zf.ZIP_DEFLATED) as zf:
    for p in sorted(OUTDIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print(f"\nArchive: {zip_path.name}  ({zip_path.stat().st_size/1024/1024:.2f} MB)")
for p in sorted(OUTDIR.glob("*")):
    print(" ", p.name)
print("\nDownload from the Kaggle Output tab and bring both CSVs back.")

HOW TO READ THIS:
- If top1_acc on 'train' is ALSO near chance (~0.0087) for a model, that model's checkpoint
  did not even memorize its own training images -- points to a training-time problem
  (bad best-epoch checkpointing, a head that never updated, etc.), not an eval-side issue.
- If 'n_unique_preds' is tiny (e.g. 1-3 out of 115) and/or 'top_pred_is_majority_class' is
  True, the model has collapsed onto predicting (close to) a constant output -- classic
  sign of a head that never learned to use its input.
- If predictions are scattered across many classes but still mostly wrong on BOTH splits,
  that rules out simple collapse but still means this checkpoint is not a working
  classifier for this task.

Archive: plantseg_r2_notebookD2_results.zip  (0.01 MB)
  vit_swin_seed42_collapse_diagnostic_predictions.csv
  vit_swin_seed42_collapse_diagnostic_summary.csv

Download from the Kaggle Output tab and bring both CSVs back.
